# KNN + Cross-Validation + Choosing the Best k

In [1]:
import pandas as pd

df = pd.DataFrame({
    "Age": [20, 22, 25, 27, 30, 32, 35, 37, 40, 42],
    "Study_Hours": [1, 2, 2, 3, 5, 6, 7, 8, 9, 10],
    "Passed": [0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
})

X = df[["Age", "Study_Hours"]]
y = df["Passed"]

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

knn_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]
)

In [4]:
k_values = [1, 3, 5, 7 ,9]

results = []

for k in k_values:

    knn_pipeline.set_params(
        model__n_neighbors = k
    )

    scores = cross_val_score(
        knn_pipeline,
        X_train,
        y_train,
        cv = 5,
        scoring="f1"
    )

    results.append({
        "k": k,
        "Mean_CV_F1": scores.mean(),
        "Std_CV_F1": scores.std()
    })

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packa

In [5]:
results_df = pd.DataFrame(results)

print(
    results_df.sort_values(
        "Mean_CV_F1",
        ascending=False
    )
)

   k  Mean_CV_F1  Std_CV_F1
0  1         1.0   0.000000
1  3         1.0   0.000000
2  5         0.8   0.163299
3  7         NaN        NaN
4  9         NaN        NaN


In [6]:
best_k = results_df.loc[
    results_df["Mean_CV_F1"].idxmax(),
    "k"
]

print("Best k:", best_k)

Best k: 1


In [7]:
final_knn = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=int(best_k)
            )
        )
    ]
)

In [8]:
final_knn.fit(
    X_train,
    y_train
)

test_prediction = final_knn.predict(X_test)

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print(
    "Accuracy:",
    accuracy_score(y_test, test_prediction)
)

print(
    "Precision:",
    precision_score(
        y_test,
        test_prediction,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        test_prediction,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        test_prediction,
        zero_division=0
    )
)

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
